# Demoparser exploration

This notebook serves as an exploration stage in demo data parsing (.dem).

In [37]:
from demoparser2 import DemoParser

sync_fields = ["round_start_time",
               "is_freeze_period",
               "round_in_progress",
               "total_rounds_played",
               "game_time",]

boundary_fields = ["is_warmup_period",
                   "round_win_status",
                   "num_player_alive_ct",
                   "num_player_alive_t",]

position_fields = ["X", "Y", "Z",
                   "player_steamid",
                   "player_name",
                   "is_alive",
                   "team_name"]

fields_to_parse = position_fields + sync_fields

parser = DemoParser("../data/raw/match-1/natus-vincere-vs-faze-m1-dust2.dem")
df = parser.parse_ticks(fields_to_parse)
print(df)

         is_freeze_period  round_start_time  total_rounds_played  \
0                    True        2201.81250                    0   
1                    True        2201.81250                    0   
2                    True        2201.81250                    0   
3                    True        2201.81250                    0   
4                    True        2201.81250                    0   
...                   ...               ...                  ...   
1724755              True        4751.84375                   18   
1724756              True        4751.84375                   18   
1724757              True        4751.84375                   18   
1724758              True        4751.84375                   18   
1724759              True        4751.84375                   18   

         round_in_progress player_name     player_steamid  team_name  \
0                    False     Twistzz  76561198016255205  TERRORIST   
1                    False     makazze 

In [ ]:
round_number = 1 # set to loop in actual script

round_df = df[df["total_rounds_played"] == round_number]

round_start_tick = round_df[round_df["is_freeze_period"] == False].iloc[0]['tick']

round_end_tick = round_df["tick"].max()

print(round_start_tick)
print(round_end_tick)

round_2_data = {
  'start_tick': round_start_tick,
  'end_tick': round_end_tick,
  'tick_rate': 64,
  'duration_ticks': round_end_tick - round_start_tick,
  'duration_seconds': (round_end_tick - round_start_tick) / 64,
}

print(round_2_data)

11280
22326
{'start_tick': np.int32(11280), 'end_tick': np.int32(22326), 'tick_rate': 64, 'duration_ticks': np.int32(11046), 'duration_seconds': np.float64(172.59375)}


In [39]:
video_round_start = 0.0  # seconds
video_round_end = 172.5   # seconds

def video_time_to_tick(video_time):
  tick = round_2_data['start_tick'] + (video_time - video_round_start) / (video_round_end - video_round_start) * (round_2_data['end_tick'] - round_2_data['start_tick'])
  return int(tick)

In [41]:
round_positions = df[
  (df['total_rounds_played'] == 1) &
  (df['is_alive'] == True)
][['tick', 'X', 'Y', 'Z', 'player_steamid', 'player_name']]

round_positions.to_parquet('../data/sample/match1_round2_positions.parquet')

metadata = {
  'round_number': round_number,
  'start_tick': round_start_tick,
  'end_tick': round_end_tick,
  'tick_rate': 64,
  'video_file': 'round-2.mp4',
  'video_round_start': video_round_start,
  'video_round_end': video_round_end
}